# Test File Summarizer - Qwen2.5-Coder-7B on Colab GPU

Generates plain-English summaries for all test files using **Qwen2.5-Coder-7B-Instruct** on Colab T4 GPU.

**No API keys. No rate limits. ~1-2 seconds per test.**

Before running: Go to **Runtime > Change runtime type > T4 GPU**

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("No GPU! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {MODEL_NAME} (4-bit quantized)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print(f"Model loaded! Memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Upload Files

Upload from your local machine:
- `unique_tests.jsonl` from `TCP-CI-dataset-v2/datasets/apache@logging-log4j2/`
- `unique_tests.jsonl` from `TCP-CI-dataset-v2/datasets/apache@shardingsphere/`

**Rename them before uploading to:**
- `log4j2_unique_tests.jsonl`
- `shardingsphere_unique_tests.jsonl`

In [ ]:
from google.colab import files
print("Upload your unique_tests.jsonl files")
print("Rename to: log4j2_unique_tests.jsonl and shardingsphere_unique_tests.jsonl")
uploaded = files.upload()

In [ ]:
import os, json
for fname in ['log4j2_unique_tests.jsonl', 'shardingsphere_unique_tests.jsonl']:
    if os.path.exists(fname):
        count = sum(1 for _ in open(fname))
        print(f"{fname}: {count} tests")
    else:
        print(f"{fname} not found - upload it above")

## Summarization Functions

In [ ]:
import json, os, time
from tqdm import tqdm

PROMPT_TEMPLATE = """Analyze this Java test file from a CI/CD pipeline.
Write a concise 2-3 sentence summary describing:
1. What functionality/component this test validates
2. What types of scenarios it covers
3. What source files/modules it likely depends on

Test file: {test_path}
```java
{source_code}
```

Respond with ONLY the summary text, nothing else."""


def generate_summary(test_path, source_code, max_source_chars=5000):
    if len(source_code) > max_source_chars:
        source_code = source_code[:max_source_chars] + "\n... [truncated]"

    prompt = PROMPT_TEMPLATE.format(test_path=test_path, source_code=source_code)
    messages = [
        {"role": "system", "content": "You are a concise code analysis assistant. Respond with only the requested summary."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=4096).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    summary = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return summary


def summarize_project(input_file, output_file):
    tests = []
    with open(input_file, encoding="utf-8") as f:
        for line in f:
            tests.append(json.loads(line))
    print(f"Loaded {len(tests)} tests from {input_file}")

    # Resume support
    already_done = set()
    if os.path.exists(output_file):
        with open(output_file, encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                already_done.add(rec["test_path"])
        print(f"Found {len(already_done)} already done (resuming)")

    remaining = [t for t in tests if t["test_path"] not in already_done]
    print(f"{len(remaining)} tests remaining")

    if not remaining:
        print("All done!")
        return

    start_time = time.time()
    with open(output_file, "a", encoding="utf-8") as out_f:
        for i, test in enumerate(tqdm(remaining, desc="Summarizing")):
            try:
                summary = generate_summary(test["test_path"], test["source_code"])
            except Exception as e:
                print(f"\n  Error on {test['test_path']}: {e}")
                summary = f"Test file at {test['test_path']}"

            record = {"test_path": test["test_path"], "summary": summary}
            out_f.write(json.dumps(record) + "\n")
            out_f.flush()

            if (i + 1) % 50 == 0:
                elapsed = time.time() - start_time
                rate = (i + 1) / elapsed
                eta = (len(remaining) - i - 1) / rate / 60
                print(f"\n  {i+1}/{len(remaining)} | {rate:.1f} tests/sec | ETA: {eta:.1f} min")

    elapsed = time.time() - start_time
    total = len(already_done) + len(remaining)
    print(f"\nDone! {total} summaries in {output_file} ({elapsed/60:.1f} min)")

## Run: log4j2 (~1,068 tests, ~20 min)

In [ ]:
summarize_project("log4j2_unique_tests.jsonl", "log4j2_test_summaries.jsonl")

## Run: shardingsphere (~3,072 tests, ~60 min)

In [ ]:
summarize_project("shardingsphere_unique_tests.jsonl", "shardingsphere_test_summaries.jsonl")

## Quality Check

In [ ]:
for fname in ['log4j2_test_summaries.jsonl', 'shardingsphere_test_summaries.jsonl']:
    if not os.path.exists(fname):
        print(f"{fname} not found yet")
        continue
    summaries = [json.loads(line) for line in open(fname)]
    lengths = [len(s['summary']) for s in summaries]
    print(f"\n=== {fname} ===")
    print(f"  Total: {len(summaries)}")
    print(f"  Avg length: {sum(lengths)/len(lengths):.0f} chars")
    print(f"  Sample:")
    for s in summaries[:3]:
        name = s['test_path'].split('/')[-1]
        print(f"    {name}: {s['summary'][:150]}...")

## Download Results

After downloading, place them in your project:
- `log4j2_test_summaries.jsonl` -> rename to `test_summaries.jsonl` in `TCP-CI-dataset-v2/datasets/apache@logging-log4j2/`
- `shardingsphere_test_summaries.jsonl` -> rename to `test_summaries.jsonl` in `TCP-CI-dataset-v2/datasets/apache@shardingsphere/`

In [ ]:
from google.colab import files
for fname in ['log4j2_test_summaries.jsonl', 'shardingsphere_test_summaries.jsonl']:
    if os.path.exists(fname):
        files.download(fname)
        print(f"Downloaded {fname}")
    else:
        print(f"{fname} not ready yet")